# Stage 0: bulk structures
This notebook does the following:
- retrieves the bulk structures by MP ID and confirms their space groups;
- selects the single-layer 1T CrTe2 cell;
- checks that corundum (111) is the c-plane;
- writes the primitive cells and `manifest.jsonl`;
- previews the VASP inputs, including explicit U and FM MAGMOM.

**Visual check:** the bulk structures and their ferromagnetic initialization.

In [ ]:
config_path = '../config/pipeline.yaml'
stage = '00_bulk'
stage_dir = '../../stages/00_bulk'

In [ ]:
import json, warnings
from pathlib import Path
import pandas as pd, yaml
from IPython.display import display
from ornl_lotf import build, mp, viz
from ornl_lotf.manifest import write_manifest
from ornl_lotf.magnetism import initial_magmoms
from ornl_lotf.vasp_driver import write_inputs
warnings.filterwarnings('ignore')
config = yaml.safe_load(Path(config_path).read_text())
out = Path(stage_dir); out.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 200, 'display.max_columns', 40)

## Retrieve and select

In [ ]:
records, sub_rows, crte2_rows = build.prepare_bulks(config, out, mp.fetch_by_ids, mp.fetch_by_formula)
print('All CrTe2 entries in Materials Project:')
crte2 = pd.DataFrame(crte2_rows); display(crte2)
print('Substrate bulks:')
subs = pd.DataFrame(sub_rows); display(subs)
crte2.to_csv(out / 'crte2_entries.csv', index=False); subs.to_csv(out / 'substrate_bulks.csv', index=False)

## Space-group confirmation
The expected space groups are listed in BUILD_OUT §6.

In [ ]:
expected = {'mp-1143': 'R-3c', 'mp-2998': 'Pm-3m', 'mp-5986': 'P4mm', 'mp-5020': 'R3m'}
for _, r in subs.iterrows():
    ok = r.spacegroup == expected.get(r.mp_id) and r.prim_spacegroup == r.spacegroup
    print(f"{r.mp_id:9s} {r.formula:7s} MP={r.spacegroup:6s} prim={r.prim_spacegroup:6s} expected={expected.get(r.mp_id)}  {'OK' if ok else 'MISMATCH'}")
    assert ok, r.mp_id
cr = [r for r in records if r.extra['material'] == 'CrTe2'][0]
print(f"CrTe2 selected: {cr.mp_id}, {cr.extra['nsites']} atoms/cell, a = {cr.extra['lattice_abc'][0]} Å (reference ≈ 3.81 Å)")

## Al2O3: rhombohedral (111) vs hexagonal (0001)

In [ ]:
al = [r for r in records if r.extra['material'] == 'Al2O3'][0]
from pymatgen.core import Structure
check = build.al2o3_c_plane_check(Structure.from_file(al.path)); print(json.dumps(check, indent=1))
assert check['same_plane'], 'rhombohedral (111) is not the hexagonal c-plane'
print('-> Stage 2 builds the Al2O3 slab as (0001) on the conventional hexagonal cell.')

## Manifest

In [ ]:
write_manifest(out, records)
display(pd.DataFrame([{'id': r.id, 'mp_id': r.mp_id, 'polymorph': r.polymorph, 'nsites': r.extra['nsites'],
                       'abc': r.extra['lattice_abc'], 'angles': r.extra['lattice_angles']} for r in records]))

## VASP input preview (explicit U on Cr; FM MAGMOM)
This uses the same `write_inputs` that the Stage 1 driver calls. The preview goes in `vasp_preview/` and is not used by Stage 1.

In [ ]:
recipe = yaml.safe_load((Path(config_path).parent / 'vasp' / 'bulk_relax.yml').read_text())
recipe['dft'] = config['dft']
rows = []
for r in records:
    wd = out / 'vasp_preview' / r.id; wd.mkdir(parents=True, exist_ok=True)
    Structure.from_file(r.path).to(filename=str(wd / 'POSCAR'), fmt='poscar')
    inc = write_inputs(wd, recipe, {'is_slab': False, 'asymmetric': False})
    rows.append({'id': r.id, 'LDAUU': inc['LDAUU'], 'LDAUL': inc['LDAUL'], 'MAGMOM': inc['MAGMOM'],
                 'ISPIN': inc['ISPIN'], 'IVDW': inc.get('IVDW'), 'ISIF': inc.get('ISIF'), 'ENCUT': inc.get('ENCUT'),
                 'KPOINTS': (wd / 'KPOINTS').read_text().splitlines()[3],
                 'POTCAR': [l.split()[3] for l in (wd / 'POTCAR').read_text().splitlines() if 'TITEL' in l]})
display(pd.DataFrame(rows))

## Structures

In [ ]:
for r in records:
    s = Structure.from_file(r.path)
    print(f"{r.id}: {s.composition.reduced_formula}, {len(s)} atoms, MAGMOM {initial_magmoms(s, config['dft']['initial_magmom'])}")
    viz.show(s, supercell=(2, 2, 2) if len(s) < 6 else (1, 1, 1))